# Notebook 07: Feed Ranking Design

## Why This Matters

Social media feeds are among the most consequential algorithmic systems ever built. The ranking decisions made by Instagram, Twitter/X, Facebook, and TikTok's FYP shape what billions of people see daily. Understanding feed ranking design is critical for social platform roles, and the design reveals deep ML challenges: training data biased by prior model decisions (position bias), engagement metrics that diverge from user wellbeing, and distribution inequality between creators.

In [ ]:
%pip install -q numpy pandas scikit-learn matplotlib

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from scipy.stats import spearmanr
np.random.seed(42)
print("Ready.")

## 1. Feed Ranking vs Recommendation

| | Recommendation | Feed Ranking |
|---|---|---|
| Freshness | Can show old items | Time decay is critical |
| Social graph | Not always relevant | First-degree connections strong signals |
| Item types | Often homogeneous | Heterogeneous: posts, stories, reels, ads |
| Candidate source | Global item pool | Social graph + interest graph + ads |

**Key difference**: Feed ranking must balance freshness, relevance, AND diversity simultaneously. A recommendation can show you a 3-year-old video. A feed showing last week's post feels broken.

In [ ]:
def generate_feed_data(n_posts=1000, n_users=100, n_obs=10000):
    posts = pd.DataFrame({
        "post_id": range(n_posts),
        "true_quality": np.random.beta(2, 5, n_posts),
        "recency_hours": np.random.exponential(24, n_posts),
        "has_video": np.random.binomial(1, 0.3, n_posts),
    })
    position_bias = {1: 1.0, 2: 0.7, 3: 0.5, 4: 0.35, 5: 0.2}
    obs = []
    for _ in range(n_obs):
        pid = np.random.randint(0, n_posts)
        pos = np.random.choice([1,2,3,4,5])
        post = posts.iloc[pid]
        recency = np.exp(-post.recency_hours / 48)
        p_click = min((post.true_quality*0.5 + recency*0.3 + post.has_video*0.1 + 0.1)*position_bias[pos], 1.0)
        obs.append({"post_id": pid, "position": pos, "clicked": np.random.binomial(1, p_click),
                    "true_quality": post.true_quality, "recency_hours": post.recency_hours, "has_video": post.has_video})
    return pd.DataFrame(obs), posts

obs_df, posts_df = generate_feed_data()
print(f"Observations: {len(obs_df)}, CTR: {obs_df.clicked.mean():.3f}")
print("CTR by position:")
print(obs_df.groupby("position")["clicked"].mean().round(3).to_string())

## 2. Position Bias and Inverse Propensity Scoring

The fundamental problem: items shown at position 1 have inflated click rates because of their position, not their quality. Training on this data teaches the model to replicate the bias.

**IPS (Inverse Propensity Scoring)** corrects by down-weighting examples shown in favorable positions:

`loss_corrected = sum (clicked_i / propensity_i) * loss(model(x_i), y_i)`

**Intuition**: A click at position 1 (high propensity) gets down-weighted because position helped. A click at position 5 (low propensity) gets up-weighted because it's a genuine preference signal despite low position.

In [ ]:
features = ["recency_hours", "has_video", "position"]
X = obs_df[features].values; y = obs_df.clicked.values
position_probs = obs_df.position.value_counts(normalize=True).sort_index()
propensities = obs_df.position.map(position_probs.to_dict()).values

X_train, X_test, y_train, y_test, prop_train, _, pos_train, pos_test, tq_train, tq_test = train_test_split(
    X, y, propensities, obs_df.position.values, obs_df.true_quality.values, test_size=0.2, random_state=42)

# Naive: include position as feature (learns bias)
model_naive = LogisticRegression(max_iter=500).fit(X_train, y_train)
naive_probs = model_naive.predict_proba(X_test)[:, 1]

# No position feature + IPS weights
X_no_pos = obs_df[["recency_hours", "has_video"]].values
X_tr_np, X_te_np, y_tr, y_te, prop_tr, _, tq_tr, tq_te = train_test_split(
    X_no_pos, y, propensities, obs_df.true_quality.values, test_size=0.2, random_state=42)

ips_weights = np.clip(1.0 / (prop_tr + 1e-6), 0.1, 20)
model_ips = LogisticRegression(max_iter=500).fit(X_tr_np, y_tr, sample_weight=ips_weights)
model_noips = LogisticRegression(max_iter=500).fit(X_tr_np, y_tr)

ips_probs = model_ips.predict_proba(X_te_np)[:, 1]
noips_probs = model_noips.predict_proba(X_te_np)[:, 1]

print("=== Position Bias Correction: Correlation with True Quality ===")
print(f"No IPS model:  {spearmanr(noips_probs, tq_te).statistic:.4f}")
print(f"IPS model:     {spearmanr(ips_probs, tq_te).statistic:.4f}")
print("IPS better correlates with true item quality (not just position-boosted quality)")

## 3. Multi-Objective Ranking and Creator Gini

Naively optimizing engagement leads to known problems:
- Maximizing clicks -> clickbait
- Maximizing watch time -> extremism rabbit holes
- Maximizing shares -> outrage content

Production systems use multi-objective scoring:
```
label = 0.4*(watch_pct >= 0.8) + 0.3*liked + 0.2*shared + 0.1*clicked - 0.5*disliked
```

**Creator Gini**: A pure engagement model over-concentrates distribution to top creators. Platforms set soft constraints on creator Gini (impression inequality) to ensure ecosystem health.

In [ ]:
def gini_coefficient(array):
    arr = np.sort(np.array(array, dtype=float))
    n = len(arr); cumulative = np.cumsum(arr)
    return (n + 1 - 2 * np.sum(cumulative) / cumulative[-1]) / n

n_creators = 100
creator_quality = np.random.pareto(1.5, n_creators); creator_quality /= creator_quality.max()

strategies = {
    "Engagement-max (amplifies inequality)": creator_quality ** 3,
    "Diversity-aware": creator_quality ** 1.2,
    "Random baseline": np.ones(n_creators),
}

print("=== Creator Distribution Gini Coefficients ===")
for name, impressions in strategies.items():
    gini = gini_coefficient(impressions)
    print(f"  {name}: Gini = {gini:.3f}")

fig, ax = plt.subplots(figsize=(8, 6))
for name, impressions in strategies.items():
    sorted_imp = np.sort(impressions); cumsum = np.cumsum(sorted_imp) / sorted_imp.sum()
    ax.plot(np.linspace(0, 1, len(cumsum)), cumsum, label=name)
ax.plot([0,1],[0,1],"k--",label="Perfect equality")
ax.set_xlabel("Cumulative fraction of creators"); ax.set_ylabel("Cumulative fraction of impressions")
ax.set_title("Lorenz Curves: Creator Distribution"); ax.legend()
plt.tight_layout(); plt.savefig("/tmp/feed_gini.png", dpi=80); plt.show()

## Summary

### Key Concepts

| Concept | Key Point |
|---|---|
| Feed freshness | Time decay required; recommendation can show old items, feed cannot |
| Position bias | Items at top get more clicks regardless of quality - biased training data |
| IPS debiasing | Weight training examples by 1/propensity to correct position bias |
| Multi-objective ranking | Combine clicks, watch time, shares, hides with explicit weights |
| Creator Gini | Measures impression inequality; pure engagement -> high Gini |

### Common Pitfalls
- Including position as a training feature (leaks position bias into model)
- Optimizing a single engagement signal
- Ignoring creator health metrics
- Not modeling the freshness constraint explicitly
